# 🔍 تدريب نموذج كشف النصوص العربية المولدة بالذكاء الاصطناعي

**الخطوات:**
1. تثبيت المكتبات
2. جمع البيانات
3. تدريب النموذج
4. تصدير ONNX
5. تحميل النموذج

**المدة المتوقعة:** ~30 دقيقة على GPU T4 المجاني

In [ ]:
# الخطوة 1: تثبيت المكتبات
!pip install -q transformers datasets accelerate scikit-learn onnx onnxruntime google-generativeai torch

In [ ]:
# الخطوة 2: جمع البيانات
# اختياري: ضع مفتاح Gemini لتوليد نصوص AI متنوعة (مجاني)
# احصل على المفتاح من: https://aistudio.google.com/apikey
import os
os.environ['GEMINI_API_KEY'] = ''  # ← ضع المفتاح هنا (اختياري)

# تحميل سكربت جمع البيانات
!wget -q https://raw.githubusercontent.com/YOUR_USERNAME/AISchool/main/training/collect_data.py
!python collect_data.py

In [ ]:
# الخطوة 3: تحميل وتجهيز البيانات
import json
import random
from pathlib import Path

data = []
with open('training_data/arabic_ai_detection_dataset.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))

random.shuffle(data)

# تقسيم: 85% تدريب، 15% اختبار
split = int(len(data) * 0.85)
train_data = data[:split]
eval_data = data[split:]

print(f'تدريب: {len(train_data)} | اختبار: {len(eval_data)}')
print(f'AI في التدريب: {sum(1 for d in train_data if d["label"]==1)}')
print(f'بشري في التدريب: {sum(1 for d in train_data if d["label"]==0)}')

In [ ]:
# الخطوة 4: تجهيز النموذج والـ Tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

MODEL_NAME = 'aubmindlab/bert-base-arabertv2'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'HUMAN', 1: 'AI'},
    label2id={'HUMAN': 0, 'AI': 1},
)

print(f'نموذج: {MODEL_NAME}')
print(f'عدد المعاملات: {model.num_parameters():,}')

In [ ]:
# الخطوة 5: تحويل البيانات لـ Dataset
train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512,
    )

train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=['text', 'source'])
eval_dataset = eval_dataset.map(tokenize_fn, batched=True, remove_columns=['text', 'source'])

train_dataset.set_format('torch')
eval_dataset.set_format('torch')

print(f'تدريب: {len(train_dataset)} | اختبار: {len(eval_dataset)}')

In [ ]:
# الخطوة 6: إعدادات التدريب
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='binary'),
        'precision': precision_score(labels, preds, average='binary'),
        'recall': recall_score(labels, preds, average='binary'),
    }

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=100,
    report_to='none',
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print('✅ جاهز للتدريب!')

In [ ]:
# الخطوة 7: التدريب 🚀
print('بدء التدريب...')
trainer.train()
print('\n✅ اكتمل التدريب!')

# تقييم نهائي
results = trainer.evaluate()
print(f'\n📊 النتائج النهائية:')
print(f'  الدقة (Accuracy): {results["eval_accuracy"]:.2%}')
print(f'  F1 Score: {results["eval_f1"]:.2%}')
print(f'  Precision: {results["eval_precision"]:.2%}')
print(f'  Recall: {results["eval_recall"]:.2%}')

In [ ]:
# الخطوة 8: حفظ النموذج
import os

SAVE_DIR = './arabic_ai_detector_v2'
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'✅ تم حفظ النموذج في {SAVE_DIR}')

In [ ]:
# الخطوة 9: تصدير ONNX
import torch
import json

ONNX_DIR = './onnx_export'
os.makedirs(ONNX_DIR, exist_ok=True)

model.eval()
dummy = tokenizer('نص تجريبي للتصدير', return_tensors='pt', padding='max_length', truncation=True, max_length=512)

onnx_path = os.path.join(ONNX_DIR, 'model.onnx')
torch.onnx.export(
    model,
    (dummy['input_ids'], dummy['attention_mask']),
    onnx_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={
        'input_ids': {0: 'batch', 1: 'seq'},
        'attention_mask': {0: 'batch', 1: 'seq'},
        'logits': {0: 'batch'},
    },
    opset_version=14,
)

# حفظ الـ tokenizer والـ labels
tokenizer.save_pretrained(ONNX_DIR)
with open(os.path.join(ONNX_DIR, 'label_map.json'), 'w') as f:
    json.dump({'0': 'HUMAN', '1': 'AI'}, f)

# التحقق
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(onnx_path)
ort_out = session.run(None, {
    'input_ids': dummy['input_ids'].numpy(),
    'attention_mask': dummy['attention_mask'].numpy(),
})

with torch.no_grad():
    pt_out = model(dummy['input_ids'], dummy['attention_mask'])

diff = abs(pt_out.logits.numpy() - ort_out[0]).max()
size_mb = os.path.getsize(onnx_path) / 1024 / 1024

print(f'✅ ONNX تم التصدير')
print(f'  الحجم: {size_mb:.1f} MB')
print(f'  الفرق: {diff:.6f}')

In [ ]:
# الخطوة 10: اختبار سريع
from transformers import pipeline

classifier = pipeline('text-classification', model=SAVE_DIR, tokenizer=SAVE_DIR)

tests = [
    ('يُعدّ الذكاء الاصطناعي من أبرز التقنيات الحديثة التي أحدثت تحولاً جذرياً في مختلف القطاعات. بالإضافة إلى ذلك فإن تطبيقاته تتوسع بشكل متسارع.', 'AI'),
    ('أنا شخصياً أحب القراءة كثير بس المشكلة إني ما ألقى وقت.. كل يوم أقول بأبدأ وما أبدأ والله حالة 😅', 'بشري'),
    ('من الجدير بالذكر أن التعليم يمثل ركيزة أساسية لبناء المجتمعات المتقدمة. علاوة على ذلك يسهم التعليم في تحقيق التنمية المستدامة.', 'AI'),
    ('لما رحت السوق أمس لقيت أسعار الخضار غالية مره.. الطماط ب٨ ريال والله حرام. قلت لأبوي وقال زمان كان الكيلو بريال', 'بشري'),
]

print('📊 اختبار النموذج:')
print('=' * 60)
for text, expected in tests:
    result = classifier(text[:512])
    label = result[0]['label']
    score = result[0]['score']
    status = '✅' if (label == 'AI' and expected == 'AI') or (label != 'AI' and expected != 'AI') else '❌'
    print(f'{status} متوقع: {expected:>5} | نتيجة: {label:>5} ({score:.1%})')
    print(f'   "{text[:80]}..."')
    print()

In [ ]:
# الخطوة 11: ضغط وتحميل 📦
!zip -r onnx_arabic_ai_detector_v2.zip onnx_export/

from google.colab import files
files.download('onnx_arabic_ai_detector_v2.zip')

print('\n✅ تم! الخطوة التالية:')
print('1. فك الضغط: unzip onnx_arabic_ai_detector_v2.zip')
print('2. انسخ المحتويات إلى: backend/analysis/onnx_model/')
print('3. ادفع التغييرات: git add . && git commit && git push')